# LAB | Whisper STT Implementation

### Ironhack - AI Integration and Consulting

## Step 1: Setting Up the Environment

Objective: Install required libraries and set up project structure.

In [1]:
print("Hello")

Hello


In [5]:
pip install openai pydub

Note: you may need to restart the kernel to use updated packages.


## Step 2: Downloading Sample Meeting Audio

Objective: Get a sample meeting audio file for transcription.

## Step 3: Transcription (Without Chunking)

Objective: Transcribe a short audio file to understand the basic API usage.

In [6]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI()

In [7]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

audio_path = r"C:\Users\antun\ai-project\W1\Day-5\lab-whisper-stt\NY045.mp3"

print("🤖 Transcribing with Whisper...")

with open(audio_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)
print("-" * 40)

🤖 Transcribing with Whisper...

📝 Transcription:
----------------------------------------
Oh, going to Cuba. I went to Cuba on Prohibition time too, and I bought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning. And, oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles. And, oh, the captain says, get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba. And the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag. So they found a $50 and threw it over the side of the b

## Step 4: Transcription with Prompts (Guided Approach)


Objective: Use prompts to guide transcription and improve accuracy.

In [8]:
from IPython.display import Audio, display, Markdown
prompt = """
This is an oral-history recording of a woman recounting her trip to Cuba during the Prohibition era. She describes buying liquor bottles as souvenirs, a ship being searched for alcohol near Atlantic City, customs inspections, fines for bringing liquor into the country, and an encounter with a doctor and a matron. Important terms and names include: Cuba, Prohibition, Atlantic City, liquor, souvenirs, ship, captain, bottles, matron, pajamas, customs inspection. """


print("-" * 40)
print("🤖 This is my prompt")
print("-" * 40)
display(Markdown(prompt))

audio_path = r"C:\Users\antun\ai-project\W1\Day-5\lab-whisper-stt\NY045.mp3"

# Checking if prompting is being used in the transcription
assert prompt.strip() != "", "Prompt is empty"

print("✅ Prompt is being used\n")

with open(audio_path, "rb") as audio_file:
    prompted_transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=prompt
    )

print("-" * 40)
print("📝 This is the Prompted transcription:")
print("-" * 40)
print(prompted_transcript.text)


----------------------------------------
🤖 This is my prompt
----------------------------------------



This is an oral-history recording of a woman recounting her trip to Cuba during the Prohibition era. She describes buying liquor bottles as souvenirs, a ship being searched for alcohol near Atlantic City, customs inspections, fines for bringing liquor into the country, and an encounter with a doctor and a matron. Important terms and names include: Cuba, Prohibition, Atlantic City, liquor, souvenirs, ship, captain, bottles, matron, pajamas, customs inspection. 

✅ Prompt is being used

----------------------------------------
📝 This is the Prompted transcription:
----------------------------------------
Oh, going to Cuba. I went to Cuba on Prohibition time, too. And I bought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it, too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning. And, oh, it was a scream. The men were coming down their pajamas and bathrobes, and they had bottles sticking all out all over. And anyone could see they had the bottles. And, oh, the captain says, get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba. And the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag. So

In [9]:
# Saving the Prompted transcript as txt file
with open("prompted_transcription.txt", "w", encoding="utf-8") as file:
    file.write(prompted_transcript.text)



## Step 5: Transcription Without Prompts (Unguided Approach)

Objective: Transcribe the same audio without prompts for comparison.

In [10]:
# Prompted transcription
with open(audio_path, "rb") as audio_file:
    prompted_transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        prompt=prompt
    )

# Unprompted transcription
with open(audio_path, "rb") as audio_file:
    unprompted_transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

print("=" * 40)
print("PROMPTED TRANSCRIPT")
print("=" * 40)
print(prompted_transcript.text)

print("=" * 40)
print("UNPROMPTED TRANSCRIPT")
print("=" * 40)
print(transcript.text)

PROMPTED TRANSCRIPT
Oh, going to Cuba. I went to Cuba on Prohibition time, too. And I bought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it, too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning. And, oh, it was a scream. The men were coming down their pajamas and bathrobes, and they had bottles sticking all out all over. And anyone could see they had the bottles. And, oh, the captain says, get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba. And the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag. So they found a $50 and threw it over the side of the boat. So then I got panicky because I had about six little bottles, and 

In [11]:
import re
import html
import difflib
from IPython.display import HTML, display

In [12]:
reference = """Oh, going to Cuba. I went to Cuba on Prohibition time too, and I bought a lot of fancy bottles, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it too. The crew had ripped the walls and put all liquor in the walls. So we all had to get up at three o'clock in the morning. And, oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles. And, oh, the captain says, get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So they looked us all over for liquor and for bringing things in from Cuba. And the girl in front of me, she had a bottle she was bringing in from her mother-in-law, and they found that bottle in her bag. So they found a $50 and threw it over the side of the boat. So then I got panicky because I had about six little bottles, and I thought, I'll have to stay on boat and wash the dishes. I spent all my money down there. I was coming home broke. So the doctors are not allowed to touch the women. They send you down to the matron. So when he went to examine me, he said, you're a very sick woman. You better go down to the matron. See, I was so nervous and upset that my pulse was going so fast, and I couldn't talk. And I was holding my coat around my stomach, see, so they wouldn't. And I guess he thought I was pregnant, see, because the bottles all started to move to the front. So he says, oh, you're like a sick woman. You better go down to the matron. And all I kept thinking about was, oh, if they ever find those bottles, I'll have to stay on ship a month to pay up for it. If they find you $50 a bottle. I think I had eight. And I was broke because I'd been down there a month, and I'd spent my money. So what happened? And she didn't examine me. She said, what's the matter? Oh, I said, I'm so nervous. I said, those doctors, they're fooling around. I said, my friend in front, they threw the bottle overboard, and I got all nervous and upset. And she says, well, I'll give you a pill, and you calm down. But I would never bring liquor in again after that. I wouldn't go through that again. For no souvenirs.
"""
    
def tokenize(text):
    """Split text into words, keeping numbers like 7.2 together."""
    return re.findall(r"\d+(?:\.\d+)?|[\w'’-]+", text)

def word_error_rate(reference, hypothesis):
    """Share of words that are wrong, missing or extra compared to the reference."""
    ref = [w.lower() for w in tokenize(reference)]
    hyp = [w.lower() for w in tokenize(hypothesis)]
    # d[i][j] = fewest edits to turn the first i reference words into the first j transcript words
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1):
        d[i][0] = i
    for j in range(len(hyp) + 1):
        d[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,         # word missing
                          d[i][j - 1] + 1,         # extra word
                          d[i - 1][j - 1] + cost)  # same or wrong word
    return d[len(ref)][len(hyp)] / len(ref)

wer_before = word_error_rate(reference, transcript.text)
wer_after = word_error_rate(reference, prompted_transcript.text)

# Every place where the two transcripts differ
before, after = tokenize(transcript.text), tokenize(prompted_transcript.text)
rows = ""
for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, before, after).get_opcodes():
    if op != "equal":
        old = html.escape(" ".join(before[i1:i2])) or "—"
        new = html.escape(" ".join(after[j1:j2])) or "—"
        rows += f"<tr><td style='color:#c0392b'>{old}</td><td style='color:#1e8449'>{new}</td></tr>"

display(HTML(f"""
<h3>Word Error Rate (lower is better)</h3>
<p>Without context: <b>{wer_before:.0%}</b> &nbsp;→&nbsp; With context: <b>{wer_after:.0%}</b></p>
<h3>Where the two transcripts differ</h3>
<table>
  <tr><th>Without context</th><th>With context</th></tr>
  {rows or "<tr><td colspan='2'>No differences</td></tr>"}
</table>
"""))

Without context,With context
and,And
in,—
and,And
If they find,They fine


Analysis:
There are differences in punctuation and sentence segmentation. The words are essentially unchanged. These changes may improve readability, but punctuation differences do not necessarily indicate better speech recognition. That might means this test is not yet showing whether contextual prompting helps with genuinely difficult words.

## Step 6, 7 and 8: Implementing Audio Chunking, Transcribing Chunks with Timestamps and Exporting with Timestamps

Objectives: 
- Split long audio files into chunks for processing.
- Transcribe each chunk and combine with proper timestamps.
- Export transcriptions in readable formats with timestamps.

In [23]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
from IPython.display import Audio, display


In [24]:
# Verifying the duration of the audio and transforming chunk to fit its lenght

from pydub import AudioSegment

audio = AudioSegment.from_mp3(audio_path)

print("Audio duration:", len(audio) / 1000 / 60, "minutos")

# 1 minute in miliseconds
chunk_length_ms = 1 * 60 * 1000

print("Each chunk size:", chunk_length_ms, "miliseconds")

chunks = [
    audio[start:start + chunk_length_ms]
    for start in range(0, len(audio), chunk_length_ms)
]

print("Total duration:", len(audio) / 1000 / 60, "minutes")
print("Quantity of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(
        f"Chunk {i + 1}: "
        f"{len(chunk) / 1000:.2f} seconds"
    )

Audio duration: 3.393733333333333 minutos
Each chunk size: 60000 miliseconds
Total duration: 3.393733333333333 minutes
Quantity of chunks: 4
Chunk 1: 60.00 seconds
Chunk 2: 60.00 seconds
Chunk 3: 60.00 seconds
Chunk 4: 23.62 seconds


In [25]:
print(f"\n🔪 Split into {len(chunks)} chunks")
print("-" * 40)

# Transcribe each chunk
all_transcripts = []

for i, chunk in enumerate(chunks):
    chunk_start = i * chunk_length_ms / 1000
    chunk_duration = len(chunk) / 1000
    chunk_end = chunk_start + chunk_duration
    
    print(f"\n🎵 Transcribing Chunk {i + 1}/{len(chunks)}...")
    print(f"Chunk duration: {len(chunk) / 1000:.2f} segundos")
    print(f"Time stemp: [{chunk_start:.2f}s - {chunk_end:.2f}s]")


    buffer = io.BytesIO()
    chunk.export(buffer, format="mp3", bitrate="64k")
    buffer.seek(0)
    buffer.name = f"chunk_{i}.mp3"

    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer,
        response_format="verbose_json",
        timestamp_granularities=["segment", "word"]
)
    
    all_transcripts.append(transcript.text)
    print(f"Chunk {i+1}: {transcript.text}")

# Full text 
print("\n📝 Complete Transcription:")
print("-" * 40)
full_text = " ".join(all_transcripts)
print(full_text)

print(f"\n🌍 Language detected: {transcript.language}")
print(f"⏱️ Total transcript duration: {transcript.duration:.2f}s")



🔪 Split into 4 chunks
----------------------------------------

🎵 Transcribing Chunk 1/4...
Chunk duration: 60.00 segundos
Time stemp: [0.00s - 60.00s]
Chunk 1: Oh, going to Cuba. I went to Cuba on Prohibition time, too, and I bought a lot of fancy bottles, a little, like one drink in for souvenirs for all my friends. Well, Atlantic City, the ship was stopped. It was all in the newspapers about it, too. The crew had ripped the walls and put all liquor in the wall. So we all had to get up at three o'clock in the morning, and oh, it was a scream. The men were coming down in their pajamas and bathrobes, and they had bottles sticking all out all over, and anyone could see they had the bottles. And oh, the captain says, Get rid of that stuff. They're searching the boats. They're downstairs ripping the wall apart. So, ah.

🎵 Transcribing Chunk 2/4...
Chunk duration: 60.00 segundos
Time stemp: [60.00s - 120.00s]
Chunk 2: They looked us all over for liquor and for bringing things in from Cuba